# Virtual EVE — Predicting Solar Irradiance from AIA Imagery

This notebook demonstrates how to use the **Virtual EVE** deep learning model to predict solar extreme ultraviolet (EUV) irradiance from SDO/AIA images.

## Background

NASA's [Solar Dynamics Observatory (SDO)](https://sdo.gsfc.nasa.gov/) carries the **EVE** (EUV Variability Experiment) instrument, which measures solar spectral irradiance across dozens of emission lines. In 2014, a capacitor failure destroyed the **MEGS-A** module, eliminating measurements of 14 key EUV spectral lines.

**Virtual EVE** restores this lost capability using deep learning. By training on four years of overlapping AIA imagery and EVE measurements (2010–2014), the model learns to predict what MEGS-A *would* have measured — effectively virtualizing the broken instrument.

The model uses a **hybrid architecture** combining:
- A **linear model** that predicts irradiance from per-channel mean/std statistics
- A **CNN** (EfficientNet-B5) that extracts spatial features from the full 512×512 px images

**Reference:** Indaco, M., Gass, D., Fawcett, W. J., Galvez, R., Wright, P. J., & Muñoz-Jaramillo, A. (2024). *Virtual EVE: a Deep Learning Model for Solar Irradiance Prediction.* [arXiv:2408.17430](https://arxiv.org/abs/2408.17430)

## What this notebook covers

1. **Loading AIA data** from the public SDOML v2 dataset on AWS S3
2. **Visualizing** multi-wavelength solar images
3. **Running inference** with the pre-trained Virtual EVE model
4. **Plotting** predicted EUV irradiance time series

## 1. Setup and Imports

Install the required packages (if not already available):

In [ ]:
%pip install -q numpy pandas matplotlib s3fs zarr torch torchvision pytorch-lightning

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import s3fs
import torch
import torchvision
import zarr
from pytorch_lightning import LightningModule
from torch import nn
from torch.nn import HuberLoss

## 2. Load AIA Data from AWS S3

The [SDOML v2 dataset](https://sdoml.org) is publicly available on AWS S3 (no credentials required). The data is stored as [Zarr](https://zarr.dev/) arrays, organized by year and wavelength.

SDO/AIA captures the Sun in 9 EUV/UV wavelength channels. Each image is 512×512 pixels.

In [ ]:
# S3 bucket and path for the SDOML v2 AIA Zarr store
S3_BUCKET = "nasa-radiant-data"
AIA_ZARR_PREFIX = "helioai-datasets/us-fdlx-ard/sdomlv2a/AIA.zarr"

# AIA wavelength channels (Angstroms)
AIA_WAVELENGTHS = ["94A", "131A", "171A", "193A", "211A", "304A", "335A", "1600A", "1700A"]

# Connect to S3 (anonymous — no credentials needed)
fs = s3fs.S3FileSystem(anon=True)
store = s3fs.S3Map(root=f"{S3_BUCKET}/{AIA_ZARR_PREFIX}", s3=fs)
aia_root = zarr.open(store, mode="r")

print("AIA Zarr store opened successfully.")
print(f"Top-level groups (years): {list(aia_root.keys())}")

### Explore the data structure

Each year group contains arrays for each wavelength. The arrays have shape `(N_timestamps, 512, 512)`. Timestamps are stored in the `T_OBS` attribute of each wavelength group.

In [ ]:
# Pick a single year to keep things fast
YEAR = "2017"

# Look at the structure for one wavelength
wl_example = "171A"
arr = aia_root[YEAR][wl_example]
print(f"Year {YEAR}, wavelength {wl_example}:")
print(f"  Shape: {arr.shape}  (N_images, height, width)")
print(f"  Dtype: {arr.dtype}")

# Read timestamps for this wavelength
t_obs_raw = fs.cat(f"{S3_BUCKET}/{AIA_ZARR_PREFIX}/{YEAR}/{wl_example}/.zattrs")
t_obs = json.loads(t_obs_raw)["T_OBS"]
print(f"  Number of timestamps: {len(t_obs)}")
print(f"  First: {t_obs[0]}")
print(f"  Last:  {t_obs[-1]}")

### Build a time index for a small date range

To keep this tutorial fast, we'll select just **6 hours** around the September 6, 2017 X9.3 solar flare — one of the most powerful flares of Solar Cycle 25. We build a simple index mapping each timestamp to its array index.

In [ ]:
# Define our time window: 6 hours around the 2017-09-06 X9.3 flare
START = pd.Timestamp("2017-09-06 09:00")
END   = pd.Timestamp("2017-09-06 15:00")

# Build a time index for the selected year
# Read T_OBS for each wavelength and find the overlapping timestamps
wl_indices = {}
for wl in AIA_WAVELENGTHS:
    raw = fs.cat(f"{S3_BUCKET}/{AIA_ZARR_PREFIX}/{YEAR}/{wl}/.zattrs")
    timestamps = json.loads(raw)["T_OBS"]
    times = pd.to_datetime(timestamps, format="mixed", utc=True).tz_localize(None)
    # Round to the 36-minute cadence used in SDOML v2
    times = times.round("36min")
    wl_indices[wl] = pd.Series(np.arange(len(times)), index=times, name=f"idx_{wl}")
    print(f"  {wl}: {len(times)} timestamps loaded")

# Join all wavelengths on their common timestamps
time_index = pd.concat(wl_indices.values(), axis=1, join="inner")
time_index = time_index[~time_index.index.duplicated(keep="first")]
time_index.sort_index(inplace=True)

# Filter to our window of interest
mask = (time_index.index >= START) & (time_index.index <= END)
time_index = time_index[mask]

print(f"\nSelected {len(time_index)} timestamps between {START} and {END}")
time_index.head()

## 3. Visualize AIA Images

Let's load and display the 9 AIA channels for a single timestamp. Each channel captures a different layer of the solar atmosphere at a different temperature.

In [ ]:
def load_aia_image(aia_root, time_index, row_idx, year="2017"):
    """Load all 9 AIA wavelength images for a given time index row."""
    row = time_index.iloc[row_idx]
    images = {}
    for wl in AIA_WAVELENGTHS:
        idx = int(row[f"idx_{wl}"])
        images[wl] = np.array(aia_root[year][wl][idx, :, :])
    return images

# Load the first timestamp in our window
aia_image = load_aia_image(aia_root, time_index, 0)
timestamp = time_index.index[0]

# Display all 9 channels
# Approximate display ranges per channel (to handle dynamic range)
VMAX = {
    "94A": 3, "131A": 20, "171A": 400, "193A": 300, "211A": 150,
    "304A": 80, "335A": 10, "1600A": 120, "1700A": 1400,
}

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle(f"SDO/AIA — {timestamp}", fontsize=14, fontweight="bold")

for ax, wl in zip(axes.flat, AIA_WAVELENGTHS):
    ax.imshow(aia_image[wl], origin="lower", cmap="inferno",
              vmin=0, vmax=VMAX.get(wl, 100))
    ax.set_title(wl, fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Define the Model Architecture

The Virtual EVE model is a **hybrid** of two sub-models:

$$O_{\text{total}} = O_{\text{linear}} + \lambda \cdot O_{\text{CNN}}$$

- The **linear model** computes the per-channel mean and standard deviation of the input images and maps them to irradiance via a single fully-connected layer.
- The **CNN model** uses an EfficientNet backbone to extract spatial features from the full images.
- A learnable parameter $\lambda$ blends the two predictions.

We define the model classes here so `torch.load` can reconstruct the checkpoint.

In [ ]:
class CNNIrradianceModel(LightningModule):
    """EfficientNet-based CNN for irradiance prediction from solar images."""

    def __init__(self, d_input, d_output, eve_norm, model="efficientnet_b0", dp=0.75):
        super().__init__()
        self.eve_norm = eve_norm

        # Load a pretrained EfficientNet and adapt it
        model_fn = getattr(torchvision.models, model)
        backbone = model_fn(weights="IMAGENET1K_V1")

        # Replace the first conv layer to accept d_input channels instead of 3 (RGB)
        conv1_out = backbone.features[0][0].out_channels
        backbone.features[0][0] = nn.Conv2d(
            d_input, conv1_out,
            kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False,
        )

        # Replace the classifier head
        lin_in = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(p=dp, inplace=True),
            nn.Linear(lin_in, d_output),
        )

        # Set dropout rate everywhere
        for m in backbone.modules():
            if m.__class__.__name__.startswith("Dropout"):
                m.p = dp

        self.model = backbone
        self.loss_func = HuberLoss()

    def forward(self, x):
        return self.model(x)


class LinearIrradianceModel(LightningModule):
    """Simple linear model: predicts irradiance from per-channel mean and std."""

    def __init__(self, d_input, d_output, eve_norm):
        super().__init__()
        self.eve_norm = eve_norm
        self.n_channels = d_input
        self.outSize = d_output
        # Input: concatenation of per-channel mean and std → 2 * n_channels features
        self.model = nn.Linear(2 * self.n_channels, self.outSize)
        self.loss_func = HuberLoss()

    def forward(self, x):
        mean_irradiance = torch.mean(x, dim=(2, 3))
        std_irradiance = torch.std(x, dim=(2, 3))
        return self.model(torch.cat((mean_irradiance, std_irradiance), dim=1))


class HybridIrradianceModel(LightningModule):
    """Hybrid model combining linear and CNN predictions."""

    def __init__(self, d_input, d_output, eve_norm, cnn_model="efficientnet_b3",
                 lr_linear=0.01, lr_cnn=0.0001, cnn_dp=0.75, ln_params=None):
        super().__init__()
        self.eve_norm = eve_norm
        self.n_channels = d_input
        self.outSize = d_output
        self.ln_params = ln_params
        self.lr_linear = lr_linear
        self.lr_cnn = lr_cnn
        self.train_mode = "linear"

        self.ln_model = LinearIrradianceModel(d_input, d_output, eve_norm)
        self.cnn_model = CNNIrradianceModel(d_input, d_output, eve_norm, model=cnn_model, dp=cnn_dp)
        self.loss_func = HuberLoss()

    def forward(self, x):
        return self.ln_model(x) + self.cnn_lambda * self.cnn_model(x)

    def forward_unnormalize(self, x):
        """Run forward pass and convert from normalized to physical units."""
        return self.unnormalize(self.forward(x))

    def unnormalize(self, y):
        eve_norm = torch.tensor(self.eve_norm).float()
        norm_mean = eve_norm[0]
        norm_stdev = eve_norm[1]
        return y * norm_stdev[None].to(y) + norm_mean[None].to(y)

    def set_train_mode(self, mode):
        if mode == "linear":
            self.train_mode = "linear"
            self.cnn_lambda = 0.0
            self.cnn_model.freeze()
            self.ln_model.unfreeze()
        elif mode == "cnn":
            self.train_mode = "cnn"
            self.cnn_lambda = 0.01
            self.cnn_model.unfreeze()
            self.ln_model.freeze()

    def training_step(self, batch, batch_nb):
        x, y = batch
        return self.loss_func(self.forward(x), y)

    def validation_step(self, batch, batch_nb):
        x, y = batch
        return self.loss_func(self.forward(x), y)

    def configure_optimizers(self):
        return torch.optim.Adam([
            {"params": self.ln_model.parameters()},
            {"params": self.cnn_model.parameters(), "lr": self.lr_cnn},
        ], lr=self.lr_linear)

## 5. Load the Pre-trained Model

The checkpoint (~42 MB) is hosted on the same public S3 bucket as the data. We download it once and cache it locally. It contains the trained model weights, the per-channel normalization statistics, and metadata about the input/output parameters.

In [ ]:
# Register model classes under the original module path so torch.load can unpickle
# (The checkpoint was saved with module paths from the training repo)
for mod_name in ["src", "src.irradiance", "src.irradiance.models", "src.irradiance.models.model"]:
    sys.modules[mod_name] = type(sys)("stub")
this_module = sys.modules[__name__]
sys.modules["src.irradiance.models.model"] = this_module

# Download checkpoint from S3 (cached locally after first run)
CHECKPOINT_S3 = "nasa-radiant-data/helioai-datasets/us-fdlx-ard/virtual-eve/AIA_MEGS_20_30_epochs_36min.ckpt"
CHECKPOINT_LOCAL = Path("AIA_MEGS_20_30_epochs_36min.ckpt")

if not CHECKPOINT_LOCAL.exists():
    print(f"Downloading checkpoint from S3 (~42 MB)...")
    fs.get(CHECKPOINT_S3, str(CHECKPOINT_LOCAL))
    print(f"Saved to {CHECKPOINT_LOCAL}")
else:
    print(f"Using cached checkpoint: {CHECKPOINT_LOCAL}")

state = torch.load(CHECKPOINT_LOCAL, map_location="cpu", weights_only=False)

model = state["model"]
model.eval()

# Extract normalization stats and scientific parameters
aia_norms = state["normalizations"]["AIA"]
wavelengths = sorted(state["sci_parameters"]["aia_wavelengths"])
eve_ions = state["sci_parameters"]["eve_ions"]

print(f"Model loaded successfully.")
print(f"Input AIA wavelengths ({len(wavelengths)}): {wavelengths}")
print(f"Output EVE ions ({len(eve_ions)}): {eve_ions[:5]} ... {eve_ions[-3:]}")

## 6. Run Inference

Now we'll run the model on each timestamp in our 6-hour window. The key steps are:

1. **Load** the 9 AIA images for each timestamp from S3
2. **Normalize** using the training-set mean and standard deviation (stored in the checkpoint)
3. **Stack** into a tensor of shape `(1, 9, 512, 512)`
4. **Forward pass** through the hybrid model to predict 38 EVE spectral lines

In [ ]:
def normalize_and_stack(aia_image, aia_norms, wavelengths):
    """Normalize raw AIA images and stack into a model-ready tensor.
    
    Args:
        aia_image: dict mapping wavelength name → (512, 512) numpy array
        aia_norms: dict mapping wavelength → {"mean": float, "std": float}
        wavelengths: sorted list of wavelength names (determines channel order)
    
    Returns:
        torch.Tensor of shape (1, n_wavelengths, 512, 512)
    """
    channels = []
    for wl in wavelengths:
        img = aia_image[wl].astype(np.float32)
        img = (img - aia_norms[wl]["mean"]) / aia_norms[wl]["std"]
        channels.append(img)
    stacked = np.stack(channels, axis=0)[np.newaxis, ...]
    return torch.from_numpy(stacked.astype(np.float32))


# Run inference on all timestamps in our window
results = []
n_total = len(time_index)

for i in range(n_total):
    ts = time_index.index[i]
    
    # Load AIA images from S3
    aia_image = load_aia_image(aia_root, time_index, i, year=YEAR)
    
    # Normalize and predict
    x = normalize_and_stack(aia_image, aia_norms, wavelengths)
    with torch.no_grad():
        pred = model.forward_unnormalize(x).numpy()[0]
    
    row = {"timestamp": ts}
    row.update({ion: float(pred[j]) for j, ion in enumerate(eve_ions)})
    results.append(row)
    
    print(f"  [{i+1}/{n_total}] {ts}", end="\r")

eve_predictions = pd.DataFrame(results).set_index("timestamp")
print(f"\nInference complete: {len(eve_predictions)} predictions for {len(eve_ions)} spectral lines.")
eve_predictions.head()

## 7. Visualize the Predicted Irradiance

Let's plot the predicted EVE irradiance time series. We'll show a selection of spectral lines to keep the plot readable, including lines from both MEGS-A (which failed in 2014) and MEGS-B (still operational, and can be used for validation).

In [ ]:
# Plot all 38 predicted EVE lines
fig, ax = plt.subplots(figsize=(14, 7))

for ion in eve_ions:
    ax.plot(eve_predictions.index, eve_predictions[ion], label=ion, alpha=0.7)

ax.set_yscale("log")
ax.set_xlabel("Time (UTC)", fontsize=12)
ax.set_ylabel("Irradiance (W/m²)", fontsize=12)
ax.set_title("Virtual EVE — Predicted Solar Irradiance (38 spectral lines)", fontsize=14)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Highlight a few key lines

With 38 lines the full plot is dense. Let's zoom in on a handful of representative ions to see the flare signature more clearly.

In [ ]:
# Select a few representative ions (pick from what's available in the predictions)
highlight_ions = [ion for ion in eve_ions if any(k in ion for k in ["Fe IX", "Fe XVI", "Fe XX", "He II", "O V"])]
if len(highlight_ions) == 0:
    highlight_ions = eve_ions[:6]  # fallback

fig, ax = plt.subplots(figsize=(14, 5))

for ion in highlight_ions:
    ax.plot(eve_predictions.index, eve_predictions[ion], label=ion, linewidth=2)

ax.set_yscale("log")
ax.set_xlabel("Time (UTC)", fontsize=12)
ax.set_ylabel("Irradiance (W/m²)", fontsize=12)
ax.set_title("Virtual EVE — Selected Ion Predictions During the 2017-09-06 X9.3 Flare", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Putting It All Together

Let's combine AIA images and irradiance predictions in a single figure to show the full pipeline: solar images in → irradiance out.

In [ ]:
# Pick the middle timestamp (likely near the flare peak)
mid_idx = len(time_index) // 2
mid_ts = time_index.index[mid_idx]
mid_image = load_aia_image(aia_root, time_index, mid_idx, year=YEAR)

fig = plt.figure(figsize=(16, 8))

# Left panel: 3x3 AIA images
for i, wl in enumerate(AIA_WAVELENGTHS):
    ax = fig.add_subplot(3, 6, (i // 3) * 6 + (i % 3) + 1)
    ax.imshow(mid_image[wl], origin="lower", cmap="inferno",
              vmin=0, vmax=VMAX.get(wl, 100))
    ax.set_title(wl, fontsize=9)
    ax.axis("off")

# Right panel: irradiance time series
ax_ts = fig.add_subplot(1, 2, 2)
for ion in highlight_ions:
    ax_ts.plot(eve_predictions.index, eve_predictions[ion], label=ion, linewidth=1.5)
ax_ts.axvline(mid_ts, color="red", linestyle="--", alpha=0.7, label=f"Shown: {mid_ts}")
ax_ts.set_yscale("log")
ax_ts.set_xlabel("Time (UTC)")
ax_ts.set_ylabel("Irradiance (W/m²)")
ax_ts.set_title("Virtual EVE Predictions")
ax_ts.legend(fontsize=8)
ax_ts.grid(True, alpha=0.3)
plt.xticks(rotation=30)

fig.suptitle(f"AIA Images → Virtual EVE Irradiance ({mid_ts})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

In this notebook we:

1. **Connected** to the public SDOML v2 dataset on AWS S3 — no credentials or local data required
2. **Loaded** multi-wavelength AIA solar images stored as Zarr arrays
3. **Ran inference** with the pre-trained Virtual EVE hybrid model (Linear + EfficientNet CNN)
4. **Predicted** 38 EVE spectral lines from 9 AIA image channels — restoring the measurements lost when MEGS-A failed in 2014

The model runs on CPU and processes each timestamp in seconds, making it practical for both research and operational use.

### Next steps

- **Extend the time range** — increase the `START`/`END` window to study longer-duration events
- **Compare with MEGS-B** — the MEGS-B module is still operational; its measurements can be used to validate the model's predictions
- **Try different events** — the September 2017 period had multiple X-class flares; try other dates to see how the model handles different solar activity levels

### Reference

Indaco, M., Gass, D., Fawcett, W. J., Galvez, R., Wright, P. J., & Muñoz-Jaramillo, A. (2024). *Virtual EVE: a Deep Learning Model for Solar Irradiance Prediction.* [arXiv:2408.17430](https://arxiv.org/abs/2408.17430)

**Data:** [SDOML v2](https://sdoml.org) — publicly available on AWS S3 (`s3://nasa-radiant-data/helioai-datasets/`)